### 整合来源不同测序技术的空转数据集

In [ ]:
import math
import os
import pandas as pd
import numpy as np

os.chdir('/pri_exthome/zhouwg/project/Garfield')
os.getcwd()

In [ ]:
# load packages
import os
import warnings
import Garfield as gf
import scanpy as sc
import numpy as np
import pandas as pd
from mudata import MuData
warnings.simplefilter(action="ignore", category=FutureWarning)
warnings.simplefilter(action='ignore', category=UserWarning)

gf.__version__

In [ ]:
# read data
import scipy.sparse as sp

dataset='mouse_olfactory_bulb'
dataset_name = 'Slide-seqV2_MoB'
root_dir = '/pri_exthome/zhouwg/project/spatial_data'
file_fold = os.path.join(root_dir, str(dataset))
adata_slide = sc.read_h5ad(file_fold + f'/{dataset_name}.h5ad')
adata_slide.obs['batch'] = 'Slide-seqV2'
adata_slide.var_names_make_unique()
adata_slide

In [ ]:
# Ensure adata.X is counts.
adata_slide.layers['counts'] = adata_slide.X.copy()
# adata.X = adata.layers['counts'].copy()
adata_slide.X.max()

In [ ]:
# read data
import scipy.sparse as sp

dataset='mouse_olfactory_bulb'
dataset_name = 'Stereo-seq_MoB'
root_dir = '/pri_exthome/zhouwg/project/spatial_data'
file_fold = os.path.join(root_dir, str(dataset))
adata_stereo = sc.read_h5ad(file_fold + f'/{dataset_name}.h5ad')
adata_stereo.obs['batch'] = 'Stereo-seq'
adata_stereo.var_names_make_unique()
adata_stereo

In [ ]:
# Ensure adata.X is counts.
adata_stereo.layers['counts'] = adata_stereo.X.copy()
# adata.X = adata.layers['counts'].copy()
adata_stereo.X.max()

In [ ]:
# import scanpy as sc
# sc.pp.normalize_total(adata_stereo, target_sum=1e4)
# sc.pp.log1p(adata_stereo)

In [ ]:
## merge data
adata = adata_slide.concatenate(adata_stereo, batch_key='Batch', index_unique=None)
adata

In [ ]:
adata.obs['batch'].value_counts()

In [ ]:
# Ensure adata.X is counts.
adata.layers['counts'] = adata.X.copy()
# adata.X = adata.layers['counts'].copy()
adata.X.max()

#### Integrating spatially resolved transcriptomics data using Garfield

In [ ]:
# set workdir #
workdir = f'/pri_exthome/zhouwg/project/Garfield_benchmark/results/sp_unimodal/spRNA_integrated'
gf.settings.set_workdir(workdir)

### modify parameter
user_config = dict(
    ## Input options
    adata_list=adata_stereo,
    profile='spatial',
    data_type='single-modal',
    sample_col='batch', # batch
    weight=0.5,
    ## Preprocessing options
    graph_const_method='mu_std', # mu_std, Radius, KNN, Squidpy
    used_hvg=True,
    min_cells=3,
    min_features=0,
    keep_mt=False,
    target_sum=1e4,
    rna_n_top_features=3000,
    n_components=50,
    n_neighbors=5,
    metric='euclidean',
    svd_solver='arpack',
    # datasets
    used_pca_feat=False,
    adj_key='connectivities',
    # data split parameters
    edge_val_ratio=0.1,
    edge_test_ratio=0.,
    node_val_ratio=0.1,
    node_test_ratio=0.,
    ## Model options
    augment_type='svd', # svd
    svd_q=5,
    use_FCencoder=True,
    conv_type='GATv2Conv', # GAT or GATv2Conv or GCN
    gnn_layer=2,
    hidden_dims=[128, 128],
    bottle_neck_neurons=20,
    cluster_num=20,
    drop_feature_rate=0.2,
    drop_edge_rate=0.2,
    num_heads=3,
    dropout=0.2,
    concat=True,
    used_edge_weight=True,
    used_DSBN=False,
    used_mmd=False,
    # data loader parameters
    num_neighbors=5,
    loaders_n_hops=2,
    edge_batch_size=4096,
    node_batch_size=512, # None
    # loss parameters
    include_edge_recon_loss=True,
    include_gene_expr_recon_loss=True,
    lambda_latent_contrastive_instanceloss=1.0,
    lambda_latent_contrastive_clusterloss=0.5,
    lambda_gene_expr_recon=1., #
    lambda_edge_recon=10., #
    lambda_latent_adj_recon_loss=2.,
    lambda_omics_recon_mmd_loss=0.5,
    # train parameters
    n_epochs_no_edge_recon=0,
    learning_rate=0.001,
    weight_decay=1e-05,
    gradient_clipping=5,
    # other parameters
    latent_key='garfield_latent',
    reload_best_model=True,
    use_early_stopping=True,
    early_stopping_kwargs=None,
    monitor=True,
    device_id=0,
    seed=2024,
    verbose=True
)
dict_config = gf.settings.set_gf_params(user_config)

In [ ]:
from Garfield.model import Garfield

# Initialize model
model = Garfield(dict_config)

In [ ]:
# Train model
model.train()

In [ ]:
# Compute latent neighbor graph
latent_key = 'garfield_latent'
sc.pp.neighbors(model.adata,
                use_rep=latent_key,
                key_added=latent_key)
# Compute UMAP embedding
sc.tl.umap(model.adata,
           neighbors_key=latent_key)

In [ ]:
# Compute latent Leiden clustering
latent_leiden_resolution = 0.3
latent_cluster_key = f"latent_leiden_{str(latent_leiden_resolution)}"
latent_key = "garfield_latent"
cell_type_key = 'cell_type'

# louvain leiden
sc.tl.leiden(adata=model.adata,
             resolution=latent_leiden_resolution,
             key_added=latent_cluster_key,
             neighbors_key=latent_key)
len(model.adata.obs[latent_cluster_key].unique())

#### Visualize Garfield Latent Space

In [ ]:
sc.settings.set_figure_params(dpi=100, facecolor='white')

sc.pl.umap(model.adata, color=['batch', latent_cluster_key],
           s=10, show=False, ncols=2, wspace=0.5) # , legend_loc='on data'

In [ ]:
import re

from matplotlib import pyplot as plt
import matplotlib
matplotlib.use("Agg") #使用非交互式的后端生成图像文件

def pic(pdf):
    searchObj = re.search( r'(.*).pdf', pdf)
    png = f"{searchObj.group(1)}.png"
    plt.savefig(pdf, bbox_inches="tight")
    plt.savefig(png, bbox_inches="tight", dpi=300)
    plt.close()

In [ ]:
import matplotlib.pyplot as plt
model.adata.obsm['spatial'][:, 1] *= -1  # 翻转 y 坐标 手动修正

In [ ]:
sc.pl.embedding(model.adata,
                basis='spatial', color=latent_cluster_key,
                title='Slide-seqV2', s=20, show=False,
                frameon=False)
# plt.gca().invert_yaxis()  # 翻转 y 轴

In [ ]:
sc.settings.set_figure_params(dpi=100, facecolor='white')

import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(8, 4), gridspec_kw={'wspace': 0.3, 'hspace': 0.2})

# model.adata.obsm['spatial'][:, 1] *= -1  # 翻转 y 坐标 手动修正
sc.pl.embedding(model.adata[model.adata.obs['batch'] == 'Slide-seqV2'],
                basis='spatial', color=latent_cluster_key,
                title='Slide-seqV2', s=20, show=False,
                ax=ax[0], frameon=False)
sc.pl.embedding(model.adata[model.adata.obs['batch'] == 'Stereo-seq'],
                basis='spatial', color=latent_cluster_key,
                title='Stereo-seq', s=20, show=False,
                ax=ax[1], frameon=False)
plt.tight_layout(w_pad=0.3)
# plt.gca().invert_yaxis()  # 翻转 y 轴
# plt.show()
# pic(os.path.join(workdir, "01.spatial_plot_niche.pdf"))

In [ ]:
## 提取特定的batch数据集
import math
import matplotlib.pyplot as plt

batch = 'Stereo-seq' # 'Slide-seqV2'
adata_batch = model.adata#[model.adata.obs['batch'] == batch].copy()
sc.pl.umap(adata_batch, color=[latent_cluster_key],
           s=10, show=False, ncols=2, wspace=0.3)

cluster_list = list(adata_batch.obs[latent_cluster_key].unique()) # ['0', '1', '3']
# cluster_list = ['7', '3', '12', '2',
#                 '6', '0', '16', '10']
highlight_color = adata_batch.uns[f'{latent_cluster_key}_colors']
default_color = 'lightgray'  # 为其他细胞设置灰色
fig, ax_list = plt.subplots(
    2, math.ceil(len(cluster_list) / 2), figsize=(2 * len(cluster_list), 8)
)
ax_list = ax_list.flatten()  # 展平以便在循环中逐一使用

tmp = adata_batch.copy()
for cluster, ax in zip(cluster_list, ax_list):
    # 设置所有细胞颜色为灰色
    tmp.obs['temp_color'] = default_color
    # 将特定 cluster 设置为高亮颜色
    cluster_cells = tmp.obs[latent_cluster_key] == cluster
    tmp.obs.loc[cluster_cells, 'temp_color'] = highlight_color[int(cluster)]

    # 绘制空间图
    sc.pl.embedding(
        tmp,
        basis='spatial',
        color='temp_color',
        palette=[highlight_color[int(cluster)], default_color],
        ax=ax,
        show=False,
        s=20,
        title=f"Niche {cluster}",
        legend_loc=None
    )

plt.tight_layout(w_pad=0.3)
# plt.show()
# pic(os.path.join(workdir, "03.spatial_plot_each_niche.pdf"))

In [ ]:

from collections import defaultdict
import scipy.sparse as sp
from scipy.sparse import isspmatrix_csr
from sklearn.preprocessing import normalize

import numpy as np
import pandas as pd
import torch
import scanpy as sc
import anndata
import anndata as ad
from anndata import AnnData, concat
from scipy.sparse import csr_matrix, hstack
from sklearn.neighbors import KNeighborsTransformer

def calc_marker_stats(ad, groupby, genes=None, use_rep='raw', inplace=False, partial=False):
    """
    Calculate marker statistics for grouped data.

    Parameters
    ----------
    ad : AnnData
        AnnData object containing expression data.
    groupby : str
        Column in `ad.obs` used for grouping cells. Must be categorical.
    genes : list, optional
        List of genes to subset for calculations. If None, all genes are used.
    use_rep : str, optional
        Which representation of data to use ('raw' or normalized). Default is 'raw'.
    inplace : bool, optional
        Whether to modify the AnnData object in place. Default is False.
    partial : bool, optional
        If True, calculate only fraction and mean statistics; skip additional computations.
        Default is False.

    Returns
    -------
    tuple or None
        If `inplace` is False, returns a tuple of DataFrames: (frac_df, mean_df, stats_df).
        Otherwise, modifies `ad` in place and returns None.
    """
    if ad.obs[groupby].dtype.name != 'category':
        raise ValueError('"%s" is not categorical' % groupby)
    n_grp = ad.obs[groupby].cat.categories.size
    if n_grp < 2:
        raise ValueError('"%s" must contain at least 2 categories' % groupby)
    # 检测是否需要标准化
    # 判断 adata.X 是否为 counts 数据（通过最大值判断）
    max_value = ad.X.max()
    if max_value >= 50:
        print(f"Detected counts data in adata.X (max value: {max_value}). Applying Scanpy normalization pipeline...")
        # 1. 总表达量归一化
        sc.pp.normalize_total(ad, target_sum=1e4)
        # 2. 对数变换
        sc.pp.log1p(ad)

    if use_rep == 'raw' and 'raw' in ad.layers.keys():
        X = ad.raw.X
        var_names = ad.raw.var_names.values
    else:
        X = ad.X
        var_names = ad.var_names.values
    if not sp.issparse(X):
        X = sp.csr_matrix(X)
    if genes:
        v_idx = var_names.isin(genes)
        X = X[:, v_idx]
        var_names = var_names[v_idx]
    # if not hasattr(ad, 'uns') or 'normalized' not in ad.uns.get('status', {}):
    #     if isspmatrix_csr(X) or X.mean() > 1 or (X.max() > 1 and X.max() < 100):
    #         # 仅在符合条件时进行标准化
    #         X = normalize(X, norm='max', axis=0)
    #         ad.uns['status'] = {'normalized': True}
    #     else:
    #         ad.uns['status'] = {'normalized': False}

    k_nonzero = X.sum(axis=0).A1 > 0
    X = X[:, np.where(k_nonzero)[0]]
    var_names = var_names[k_nonzero]

    n_var = var_names.size
    x = np.arange(n_var)

    grp_indices = {k: g.index.values for k, g in ad.obs.reset_index().groupby(groupby, sort=False)}

    frac_df = pd.DataFrame({k: (X[idx, :] > 0).mean(axis=0).A1 for k, idx in grp_indices.items()}, index=var_names)
    mean_df = pd.DataFrame({k: X[idx, :].mean(axis=0).A1 for k, idx in grp_indices.items()}, index=var_names)

    if partial:
        stats_df = None
    else:
        frac_order = np.apply_along_axis(np.argsort, axis=1, arr=frac_df.values)
        y1 = frac_order[:, n_grp - 1]
        y2 = frac_order[:, n_grp - 2]
        y3 = frac_order[:, n_grp - 3] if n_grp > 2 else y2
        top_frac_grps = frac_df.columns.values[y1]
        top_fracs = frac_df.values[x, y1]
        frac_diffs = top_fracs - frac_df.values[x, y2]
        max_frac_diffs = top_fracs - frac_df.values[x, y3]

        mean_order = np.apply_along_axis(np.argsort, axis=1, arr=mean_df.values)
        y1 = mean_order[:, n_grp - 1]
        y2 = mean_order[:, n_grp - 2]
        y3 = mean_order[:, n_grp - 3] if n_grp > 2 else y2
        top_mean_grps = mean_df.columns.values[y1]
        top_means = mean_df.values[x, y1]
        mean_diffs = top_means - mean_df.values[x, y2]
        max_mean_diffs = top_means - mean_df.values[x, y3]

        stats_df = pd.DataFrame({
            'top_frac_group': top_frac_grps, 'top_frac': top_fracs, 'frac_diff': frac_diffs,
            'max_frac_diff': max_frac_diffs,
            'top_mean_group': top_mean_grps, 'top_mean': top_means, 'mean_diff': mean_diffs,
            'max_mean_diff': max_mean_diffs
        }, index=var_names)
        stats_df['top_frac_group'] = stats_df['top_frac_group'].astype('category')
        stats_df['top_frac_group'] = stats_df['top_frac_group'].cat.reorder_categories(
            list(ad.obs[groupby].cat.categories))

    if inplace:
        if use_rep == 'raw':
            ad.raw.varm[f'frac_{groupby}'] = frac_df
            ad.raw.varm[f'mean_{groupby}'] = mean_df
            if not partial:
                ad.raw.var = pd.concat([ad.raw.var, stats_df], axis=1)
        else:
            ad.varm[f'frac_{groupby}'] = frac_df
            ad.varm[f'mean_{groupby}'] = mean_df
            if not partial:
                ad.var = pd.concat([ad.raw.var, stats_df], axis=1)
    else:
        return frac_df, mean_df, stats_df


In [ ]:

def filter_marker_stats(data, use_rep='raw', min_frac_diff=0.1, min_mean_diff=0.1, max_next_frac=0.9,
                        max_next_mean=0.95, strict=False, how='or'):
    """
    Filter marker statistics based on thresholds.

    Parameters
    ----------
    data : AnnData or DataFrame
        Data containing marker statistics.
    use_rep : str, optional
        Which data representation to use ('raw' or processed). Default is 'raw'.
    min_frac_diff : float, optional
        Minimum difference in fraction to consider a marker valid. Default is 0.1.
    min_mean_diff : float, optional
        Minimum difference in mean expression to consider a marker valid. Default is 0.1.
    max_next_frac : float, optional
        Maximum fraction difference for non-top markers. Default is 0.9.
    max_next_mean : float, optional
        Maximum mean difference for non-top markers. Default is 0.95.
    strict : bool, optional
        If True, use stricter thresholds for filtering. Default is False.
    how : str, optional
        Logical operation to combine fraction and mean filters ('or' or 'and'). Default is 'or'.

    Returns
    -------
    DataFrame
        Filtered DataFrame containing marker statistics.
    """
    columns = ['top_frac_group', 'top_frac', 'frac_diff', 'max_frac_diff', 'top_mean_group', 'top_mean', 'mean_diff',
               'max_mean_diff']
    if isinstance(data, anndata.AnnData):
        stats_df = data.raw.var[columns] if use_rep == 'raw' else data.var[columns]
    elif isinstance(data, pd.DataFrame):
        stats_df = data[columns]
    else:
        raise ValueError('Invalid input, must be an AnnData or DataFrame')
    frac_diff = stats_df.frac_diff if strict else stats_df.max_frac_diff
    mean_diff = stats_df.mean_diff if strict else stats_df.max_mean_diff
    same_group = stats_df.top_frac_group == stats_df.top_mean_group
    meet_frac_requirement = (frac_diff >= min_frac_diff) & (stats_df.top_frac - frac_diff <= max_next_frac)
    meet_mean_requirement = (mean_diff >= min_mean_diff) & (stats_df.top_mean - mean_diff <= max_next_mean)
    if how == 'or':
        filtered = stats_df.loc[same_group & (meet_frac_requirement | meet_mean_requirement)]
    else:
        filtered = stats_df.loc[same_group & (meet_frac_requirement & meet_mean_requirement)]
    if strict:
        filtered = filtered.sort_values(['top_frac_group', 'mean_diff', 'frac_diff'], ascending=[True, False, False])
    else:
        filtered = filtered.sort_values(['top_frac_group', 'mean_diff', 'frac_diff'], ascending=[True, False, False])
    filtered['top_frac_group'] = filtered['top_frac_group'].astype('category')
    filtered['top_frac_group'] = filtered['top_frac_group'].cat.reorder_categories(
        list(stats_df['top_frac_group'].cat.categories))
    return filtered


def plot_markers(
        adata: anndata.AnnData,
        groupby: str,
        mks: pd.DataFrame,
        n_genes: int = 5,
        kind: str = 'dotplot',
        remove_genes: list = [],
        **kwargs
):
    """
    Plot markers for specific groups.

    Parameters
    ----------
    adata : AnnData
        AnnData object containing expression data.
    groupby : str
        Column in `adata.obs` used for grouping cells.
    mks : DataFrame
        DataFrame containing marker statistics.
    n_genes : int, optional
        Number of top genes to plot per group. Default is 5.
    kind : str, optional
        Type of plot to create ('dotplot', 'violin', etc.). Default is 'dotplot'.
    remove_genes : list, optional
        List of genes to exclude from the plot. Default is an empty list.
    **kwargs : dict
        Additional keyword arguments passed to the plotting function.

    Returns
    -------
    matplotlib.Axes or None
        Axes object of the plot or None if plotting in place.
    """
    df = mks.reset_index()[['index', 'top_frac_group']].rename(columns={'index': 'gene', 'top_frac_group': 'cluster'})
    var_tb = adata.raw.var if kwargs.get('use_raw', None) == True or adata.raw else adata.var
    remove_gene_set = set()
    for g_cat in remove_genes:
        if g_cat in var_tb.columns:
            remove_gene_set |= set(var_tb.index[var_tb[g_cat].values])
    df = df[~df.gene.isin(list(remove_gene_set))].copy()
    df1 = df.groupby('cluster').head(n_genes)
    mks_dict = defaultdict(list)
    for c, g in zip(df1.cluster, df1.gene):
        mks_dict[c].append(g)
    func = getattr(sc.pl, kind)
    if sc.__version__.startswith('1.4'):
        return func(adata, df1.gene.to_list(), groupby=groupby, **kwargs)
    else:
        return func(adata, mks_dict, groupby=groupby, **kwargs)


def aggregate_top_markers(ad, mks, groupby, n_genes=100, use_raw=False, **kwargs):
    """
    Aggregate top marker genes.

    Parameters
    ----------
    ad : AnnData
        AnnData object containing expression data.
    mks : DataFrame
        DataFrame containing marker statistics.
    groupby : str
        Column in `ad.obs` used for grouping cells.
    n_genes : int, optional
        Number of top genes to include in the test. Default is 100.
    use_raw : bool, optional
        Whether to use raw expression values for the test. Default is True.
    **kwargs : dict
        Additional keyword arguments passed to the rank_genes_groups function.

    Returns
    -------
    DataFrame
        Merged DataFrame with marker statistics and differential expression results.
    """
    genes = top_markers(mks, top_n=n_genes)
    aux_ad = anndata.AnnData(
        X=ad.raw.X if use_raw else ad.X,
        obs=ad.obs.copy(),
        var=ad.raw.var.copy() if use_raw else ad.var.copy()
    )
    aux_ad = aux_ad[:, genes].copy()
    sc.tl.rank_genes_groups(aux_ad, groupby=groupby, n_genes=n_genes, use_raw=False, **kwargs)
    de_tbl = extract_de_table(aux_ad.uns['rank_genes_groups'])
    return mks.reset_index().rename(columns={'index': 'genes', 'top_frac_group': 'cluster'}).merge(
        de_tbl[['cluster', 'genes', 'logfoldchanges', 'pvals', 'pvals_adj']], how='left'
    )


def extract_de_table(de_dict):
    """
    Extract a differential expression table from an AnnData.uns dictionary.

    Parameters
    ----------
    de_dict : dict
        Dictionary containing differential expression results from AnnData.

    Returns
    -------
    DataFrame
        DataFrame containing differential expression statistics, including cluster, rank,
        gene names, and relevant metrics (e.g., logfoldchanges, p-values).
    """
    if de_dict['params']['method'] == 'logreg':
        requested_fields = ('scores',)
    else:
        requested_fields = ('scores', 'logfoldchanges', 'pvals', 'pvals_adj',)
    gene_df = _recarray_to_dataframe(de_dict['names'], 'genes')[
        ['cluster', 'rank', 'genes']]
    gene_df['ref'] = de_dict['params']['reference']
    gene_df = gene_df[['cluster', 'ref', 'rank', 'genes']]
    de_df = pd.DataFrame({
        field: _recarray_to_dataframe(de_dict[field], field)[field]
        for field in requested_fields if field in de_dict
    })
    de_tbl = gene_df.merge(de_df, left_index=True, right_index=True)
    de_tbl = de_tbl.loc[de_tbl.genes.astype(str) != 'nan', :]
    return de_tbl


def _recarray_to_dataframe(array, field_name):
    return pd.DataFrame(array).reset_index().rename(
        columns={'index': 'rank'}).melt(
        id_vars='rank', var_name='cluster', value_name=field_name)


def top_markers(df, top_n=5, groupby='top_frac_group'):
    return df.groupby(groupby).head(top_n).index.to_list()


In [ ]:
### 寻找 niche specific 的marker
# from Garfield.model.utils import calc_marker_stats, filter_marker_stats, plot_markers

latent_leiden_resolution = 0.3
latent_cluster_key = f"latent_leiden_{str(latent_leiden_resolution)}"

adata_stereo.obs[latent_cluster_key] = model.adata.obs[latent_cluster_key]
mkst = calc_marker_stats(adata_stereo, groupby=latent_cluster_key)


In [ ]:
mks = filter_marker_stats(mkst[2], min_frac_diff=0, min_mean_diff=0, max_next_frac=0.9, max_next_mean=0.95, strict=False, how='or')
plot_markers(adata_stereo, groupby=latent_cluster_key, mks=mks)

In [ ]:
df = aggregate_top_markers(adata_stereo, mks, groupby=latent_cluster_key,
                           n_genes=100, use_raw=False, method="wilcoxon")

#### 找 niche 差异基因

In [ ]:
latent_leiden_resolution = 0.3
latent_cluster_key = f"latent_leiden_{str(latent_leiden_resolution)}"

adata_stereo.obs[latent_cluster_key] = model.adata.obs[latent_cluster_key]
sc.tl.rank_genes_groups(adata_stereo, latent_cluster_key, method='wilcoxon')

In [ ]:
df  = sc.get.rank_genes_groups_df(adata_stereo, group=None)
df = df.sort_values(by="scores", ascending=False)
df.to_csv(os.path.join(workdir, 'diff_niches_stereo.csv'), index=0, sep='\t')

In [ ]:
def extract_top_genes_spatial(
    adata,
    workdir,
    latent_cluster_key,
    n_top_markers=5,
    output_file="all_marker_genes_spatial_data.csv",
    use_normalized=False
):
    """
    提取 AnnData 对象中所有 marker 基因的表达数据、空间位置信息和 cluster 信息，并保存为 CSV 文件。

    Parameters:
    -----------
    adata : AnnData
        包含 rank_genes_groups 和空间信息的 AnnData 对象。
    workdir : str
        输出文件的保存目录。
    latent_cluster_key : str
        cluster 信息所在的 obs 键。
    output_file : str
        保存的 CSV 文件名。
    use_normalized : bool
        如果为 True，使用标准化后的数据 (.X)；否则使用 counts 层数据。

    Returns:
    --------
    None
    """
    import os
    import pandas as pd

    # 检查 rank_genes_groups 是否存在
    if "rank_genes_groups" not in adata.uns:
        raise ValueError("The AnnData object does not contain 'rank_genes_groups' in .uns.")

    # 提取所有 marker 基因
    ranked_genes = adata.uns['rank_genes_groups']
    marker_genes = set()  # 使用集合避免重复
    clusters = ranked_genes['names'].dtype.names  # 获取所有 cluster 名
    for cluster in clusters:
        top_genes = ranked_genes['names'][cluster][:n_top_markers]  # 获取Top n 基因名
        print(f'Top markers of cluster {cluster}: {top_genes}')
        marker_genes.update(top_genes)  # 添加当前 cluster 的 marker 基因

    marker_genes = list(marker_genes)  # 转换为列表

    # 提取表达数据
    if use_normalized:
        gene_expression = adata[:, marker_genes].X  # 标准化数据
    else:
        gene_expression = adata[:, marker_genes].layers['counts']  # counts 数据

    # 提取空间位置信息
    spatial_positions = adata.obsm['spatial']

    # 提取 cluster 信息
    clusters = adata.obs[latent_cluster_key]

    # 构建最终 DataFrame
    final_dataframe = pd.DataFrame(
        gene_expression.toarray() if hasattr(gene_expression, "toarray") else gene_expression,
        columns=marker_genes,
        index=adata.obs.index
    )
    final_dataframe['x'] = spatial_positions[:, 0]  # 添加 x 坐标
    final_dataframe['y'] = spatial_positions[:, 1]  # 添加 y 坐标
    final_dataframe['cluster'] = clusters.values  # 添加 cluster 信息

    # 保存为 CSV 文件
    os.makedirs(workdir, exist_ok=True)  # 确保工作目录存在
    final_dataframe.to_csv(os.path.join(workdir, output_file))
    print(f"Data saved to {os.path.join(workdir, output_file)}")


In [ ]:
## output res
extract_top_genes_spatial(
    adata_stereo,
    workdir,
    latent_cluster_key,
    n_top_markers=5,
    output_file="top_genes_spatial_data_Stereo-seq_MoB.csv",
    use_normalized=False
)

In [ ]:
## niche 注释
cluster2annotation = {
    '0': 'GCL_external', # Pcp4
    '1': 'GCL_deep', # Nrxn3
    '2': 'Cpe+ niche', # Cpe
    '3': 'Ap3s1+ niche', # Ap3s1
    '4': 'ONL', # Apod
    '5': 'ONL',
    '6': 'EPL', # Slc6a11
    '7': 'IPL', # Slc17a7
    '8': 'GCL_deep', # Nrxn3
    '9': 'GL', # Cck
    '10': 'RMS', # Mbp
    '11': 'Gad1+ niche',
    '12': 'MCL', # Vip
    '13': 'ONL',
    '14': 'ONL'
}
model.adata.obs['niche_type'] = model.adata.obs[latent_cluster_key].map(cluster2annotation).astype('category')

In [ ]:
# gene_list = ['Ptgds', 'S100a5', 'Mgst1', 'Apold1', 'Ly6g6e',
#              'Cbln4', 'Vip', 'Slc17a7', 'Tpbg', 'Penk', 'Sox11']

gene_list = ["Vip", "Olfm1", "Ptgds",
             "Pcp4",  "Camk2b", "Calb2",
             "Gad1", "S100a5", "Plp1"]
fig, ax_list = plt.subplots(
    2, len(gene_list) // 2, figsize=(2 * len(gene_list), 8)
)
ax_list = ax_list.flatten()  # 展平以便在循环中逐一使用

for gene, ax in zip(gene_list, ax_list):
    sc.pl.embedding(adata_stereo, basis='spatial', color=gene,
                    ax=ax, show=False, color_map='YlOrRd', s=20)

plt.tight_layout(w_pad=0.3)
# plt.show()
# pic(os.path.join(workdir, "04.spatial_plot_each_niche_marker.pdf"))

In [ ]:
marker_genes_dict = {
    'ONL': ['Apod'],
    'GL': ['Cck'],
    'EPL': ['Slc6a11'],
    'MCL': ['Vip'],
    'IPL': ['Slc17a7'],
    'GCL_external': ['Pcp4'],
    'GCL_deep': ['Nrxn3'],
    'RMS': ['Mbp'],
    'Cpe+ niche': ['Cpe'],
    'Ap3s1+ niche': ['Ap3s1'],
    'Gad1+ niche': ['Gad1']
}
tmp = adata_stereo.copy()
tmp.obs['niche_type'] = model.adata.obs['niche_type']

# 修改因子顺序
tmp.obs['niche_type'].cat.categories
tmp.obs['niche_type'] = tmp.obs['niche_type'].cat.reorder_categories(['ONL', 'GL', 'EPL',
                                                                      'MCL', 'IPL','GCL_external', 'GCL_deep', 'RMS','Cpe+ niche', 'Ap3s1+ niche', 'Gad1+ niche'], ordered=True)
# tmp.obs['niche_type'].cat.categories
sc.pl.dotplot(tmp, marker_genes_dict, 'niche_type',
              dendrogram=False, standard_scale='var')

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# 确认cluster信息是否存在
cluster_key = latent_cluster_key  # 替换为实际的cluster列名
adata_stereo.obs[latent_cluster_key] = model.adata.obs[latent_cluster_key]
if cluster_key not in adata_stereo.obs:
    raise ValueError(f"{cluster_key} not found in adata.obs")

# 提取 marker 基因列表
# marker_genes = [gene for genes in marker_genes_dict.values() for gene in (genes if isinstance(genes, list) else [genes])]

marker_genes = ["Vip", "Olfm1", "Ptgds",
             "Pcp4",  "Camk2b", "Calb2",
             "Gad1", "S100a5", "Plp1"]

# 确保基因在adata.var_names中
filtered_genes = [gene for gene in marker_genes if gene in adata_stereo.var_names]
missing_genes = set(marker_genes) - set(filtered_genes)
if missing_genes:
    print(f"Warning: The following genes are missing in adata.var_names: {missing_genes}")

# Subset表达矩阵
adata_subset = adata_stereo[:, filtered_genes].to_df()

# 添加cluster信息
adata_subset['cluster'] = adata_stereo.obs[cluster_key].values

# 按cluster计算平均表达
mean_expression = adata_subset.groupby('cluster').mean()

# 绘制热图
plt.figure(figsize=(10, len(filtered_genes) * 0.4))
sns.heatmap(mean_expression.T, annot=True, fmt=".2f", cmap='viridis', cbar_kws={'label': 'Mean Expression'})
plt.title("Average Expression of Marker Genes Across Clusters")
plt.xlabel("Cluster")
plt.ylabel("Marker Genes")
plt.tight_layout()
plt.show()

#### 保存model 结果

In [ ]:
# Save trained model
model_folder_path = f"{workdir}/model_stereo"
os.makedirs(model_folder_path, exist_ok=True)

model.save(dir_path=model_folder_path,
           overwrite=True,
           save_adata=True,
           adata_file_name="adata_stereo.h5ad")

In [ ]:
from Garfield.model import Garfield

workdir = f'/pri_exthome/zhouwg/project/Garfield_benchmark/results/sp_unimodal/spRNA_integrated'
gf.settings.set_workdir(workdir)
model_folder_path = f"{workdir}/model_stereo"

model = Garfield.load(dir_path=model_folder_path,
              adata_file_name="adata_stereo.h5ad")

In [ ]:
import numpy as np
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from tqdm import trange
from scipy.spatial import *
from sklearn.preprocessing import *

from sklearn.metrics import *
from scipy.spatial.distance import *

def CHAOS_score(X, pred_labels):
    """
    Calculate the CHAOS score for a given set of spatial coordinates and predicted labels.

    param: X - spatial coordinates
    param: pred_labels - predicted labels

    return: CHAOS score
    """
    # Standardize the spatial coordinates
    X = StandardScaler().fit_transform(X)

    # Get the unique cluster labels
    cluster_labels = np.unique(pred_labels)

    # Initialize the distance value and count
    dist_val = 0.
    count = 0

    # Iterate through each cluster
    for k in cluster_labels:
        # Get the spatial coordinates for the current cluster
        cluster_coords = X[pred_labels == k, :]

        # Check if there are at least 2 spatial coordinates in the cluster
        if len(cluster_coords) <= 2:
            continue
        else:
            count += len(cluster_coords)

        # Calculate the distance to the nearest neighbor for each spatial coordinate in the cluster
        nbrs = NearestNeighbors(n_neighbors=1).fit(cluster_coords)
        distances, _ = nbrs.kneighbors()

        # Sum the distances
        dist_val = dist_val + np.sum(distances)

    # Calculate the CHAOS score
    return dist_val / count

In [ ]:
import pandas as pd

# Define a function to compute CHAOS score for a range of resolutions
def compute_chaos_scores(adata, resolutions, latent_key="garfield_latent"):
    results = []
    for resolution in resolutions:
        latent_cluster_key = f"latent_leiden_{resolution}"

        # Perform Leiden clustering
        sc.tl.leiden(adata=adata,
                      resolution=resolution,
                      key_added=latent_cluster_key,
                      neighbors_key=latent_key)

        # Compute CHAOS score
        X = adata.obsm['spatial']
        pred_labels = adata.obs[latent_cluster_key].values
        chaos = CHAOS_score(X=X, pred_labels=pred_labels)

        # Store results
        results.append({"Resolution": resolution, "N_niches": len(model.adata.obs[latent_cluster_key].unique()),"CHAOS_Score": chaos})

    # Convert results to a DataFrame
    df_results = pd.DataFrame(results)
    return df_results

# Define a list of resolutions
resolutions = [0.1, 0.2, 0.3, 0.4, 0.5]

# Call the function with your adata object and resolutions
# Assuming `model.adata` is the AnnData object used
chaos_results_df = compute_chaos_scores(adata=model.adata, resolutions=resolutions)

# Save the results for R visualization
chaos_results_df.to_csv(os.path.join(workdir, "chaos_results_stereo.csv"), index=False)

In [ ]:
chaos_results_df

In [ ]:
### 输出niche 和 空间位置数据供R绘图
# Extracting 'latent_leiden_0.3' and 'spatial' data
visua_type = 'niche_type'
latent_leiden_data = model.adata.obs[visua_type] # latent_cluster_key
spatial_data = model.adata.obsm["spatial"]

# Creating a DataFrame for R visualization
output_df = pd.DataFrame({
    visua_type: latent_leiden_data,
    "spatial_X": spatial_data[:, 0],
    "spatial_Y": spatial_data[:, 1],
})

# Save to CSV
output_file = f"latent_spatial_data_{dataset_name}.csv"
output_df.to_csv(os.path.join(workdir, output_file))

In [ ]:
### 输出niche 和 UMAP位置数据供R绘图
# Extracting 'latent_leiden_0.3' and 'spatial' data
visua_type = 'niche_type'
latent_leiden_data = model.adata.obs[visua_type] # latent_cluster_key
spatial_data = model.adata.obsm["X_umap"]

# Creating a DataFrame for R visualization
output_df = pd.DataFrame({
    visua_type: latent_leiden_data,
    "umap_1": spatial_data[:, 0],
    "umap_2": spatial_data[:, 1],
})

# Save to CSV
output_file = f"latent_umap_data_{dataset_name}.csv"
output_df.to_csv(os.path.join(workdir, output_file))

In [ ]:
dataset_name

### 空转数据之间 niches transfer

In [ ]:
from Garfield.model import Garfield

workdir = f'/pri_exthome/zhouwg/project/Garfield_benchmark/results/sp_unimodal/spRNA_integrated'
gf.settings.set_workdir(workdir)
model_folder_path = f"{workdir}/model_stereo"

model = Garfield.load(dir_path=model_folder_path,
              adata_file_name="adata_stereo.h5ad")

In [ ]:
adata_stereo

In [ ]:
# read spatial data
import scipy.sparse as sp

dataset='model_slide'
dataset_name = 'slide'
root_dir = '/pri_exthome/zhouwg/project/Garfield_benchmark/results/sp_unimodal/spRNA_integrated'
file_fold = os.path.join(root_dir, str(dataset))
tmp = sc.read_h5ad(file_fold + f'/adata_{dataset_name}.h5ad')
query_adata = adata_slide.copy()
query_adata.obs['latent_leiden_0.5'] = tmp.obs['latent_leiden_0.5']
query_adata.var_names_make_unique()
query_adata

In [ ]:
# Ensure adata.X is counts.
# query_adata.layers['counts'] = query_adata.X.copy()
query_adata.X = query_adata.layers['counts'].copy()
query_adata.X.max()

#### Integrating spatially resolved transcriptomics data using Garfield

In [ ]:
new_model = model.load_query_data(dir_path=model_folder_path,
                                  query_adata=query_adata,
                                  ref_adata_name="adata_stereo.h5ad",
                                  use_cuda=True,
                                  unfreeze_all_weights=False,
                                  unfreeze_eps_weight=True,
                                  unfreeze_layer0=True,
                                  used_mmd=True,
                                  sample_col='projection',
                                  lambda_omics_recon_mmd_loss=1.0)
# Training and obtain latent representation
new_model.train()

# plot UMAP
sc.pp.neighbors(new_model.adata, use_rep='garfield_latent')
sc.tl.umap(new_model.adata)
sc.pl.umap(new_model.adata,
           color=['projection', 'latent_leiden_0.5','niche_type'],
           ncols=1, wspace=0.20, edges=False)

In [ ]:
## split
adata_ref = new_model.adata[new_model.adata.obs['projection'] == 'reference', :]
adata_query = new_model.adata[new_model.adata.obs['projection'] == 'query', :]
spatial_tmp = query_adata.obsm['spatial'].copy()

In [ ]:
### Label transfer
## major celltype
adata_query = new_model.label_transfer(ref_adata=adata_ref,
                                       ref_adata_emb='garfield_latent',
                                       query_adata=adata_query,
                                       query_adata_emb='garfield_latent',
                                       n_neighbors=5,
                                       ref_adata_obs=adata_ref.obs,
                                       label_keys='niche_type')

In [ ]:
adata_query

In [ ]:
adata_query.obsm['spatial'] = spatial_tmp
import matplotlib.pyplot as plt
adata_query.obsm['spatial'][:, 1] *= -1  # 翻转 y 坐标 手动修正

sc.settings.set_figure_params(dpi=100, facecolor='white')
sc.pl.embedding(adata_query, basis="spatial",
                color=["transferred_niche_type_unfiltered"],
                ncols=1, wspace=0.20, edges=False)

sc.pl.umap(adata_query, color=["transferred_niche_type_unfiltered"],
           ncols=1, wspace=0.20, edges=False)

In [ ]:
cluster_list

In [ ]:
## 提取特定的batch数据集
import math
import matplotlib.pyplot as plt

latent_cluster_key = 'transferred_niche_type_unfiltered'

cluster_list = list(adata_query.obs[latent_cluster_key].unique()) # ['0', '1', '3']
# cluster_list = ['7', '3', '12', '2',
#                 '6', '0', '16', '10']
highlight_color = adata_query.uns[f'{latent_cluster_key}_colors']
default_color = 'lightgray'  # 为其他细胞设置灰色
fig, ax_list = plt.subplots(
    2, math.ceil(len(cluster_list) / 2), figsize=(2 * len(cluster_list), 8)
)
ax_list = ax_list.flatten()  # 展平以便在循环中逐一使用

tmp = adata_query.copy()
for cluster, ax in zip(cluster_list, ax_list):
    # 设置所有细胞颜色为灰色
    tmp.obs['temp_color'] = default_color
    # 将特定 cluster 设置为高亮颜色
    cluster_cells = tmp.obs[latent_cluster_key] == cluster
    cluster_index = cluster_list.index(cluster)
    tmp.obs.loc[cluster_cells, 'temp_color'] = highlight_color[cluster_index]
    # tmp.obs.loc[cluster_cells, 'temp_color'] = highlight_color[int(cluster)]

    # 绘制空间图
    sc.pl.embedding(
        tmp,
        basis='spatial',
        color='temp_color',
        palette=[highlight_color[cluster_index], default_color],
        ax=ax,
        show=False,
        s=20,
        title=f"Niche {cluster}",
        legend_loc=None
    )

plt.tight_layout(w_pad=0.3)
# plt.show()
# pic(os.path.join(workdir, "03.spatial_plot_each_niche.pdf"))

In [ ]:
## 提取特定的batch数据集
import math
import matplotlib.pyplot as plt

latent_cluster_key = 'latent_leiden_0.5'

cluster_list = list(adata_query.obs[latent_cluster_key].unique()) # ['0', '1', '3']
# cluster_list = ['7', '3', '12', '2',
#                 '6', '0', '16', '10']
highlight_color = adata_query.uns[f'{latent_cluster_key}_colors']
default_color = 'lightgray'  # 为其他细胞设置灰色
fig, ax_list = plt.subplots(
    2, math.ceil(len(cluster_list) / 2), figsize=(2 * len(cluster_list), 8)
)
ax_list = ax_list.flatten()  # 展平以便在循环中逐一使用

tmp = adata_query.copy()
for cluster, ax in zip(cluster_list, ax_list):
    # 设置所有细胞颜色为灰色
    tmp.obs['temp_color'] = default_color
    # 将特定 cluster 设置为高亮颜色
    cluster_cells = tmp.obs[latent_cluster_key] == cluster
    cluster_index = cluster_list.index(cluster)
    tmp.obs.loc[cluster_cells, 'temp_color'] = highlight_color[cluster_index]
    # tmp.obs.loc[cluster_cells, 'temp_color'] = highlight_color[int(cluster)]

    # 绘制空间图
    sc.pl.embedding(
        tmp,
        basis='spatial',
        color='temp_color',
        palette=[highlight_color[cluster_index], default_color],
        ax=ax,
        show=False,
        s=20,
        title=f"Niche {cluster}",
        legend_loc=None
    )

plt.tight_layout(w_pad=0.3)
# plt.show()
# pic(os.path.join(workdir, "03.spatial_plot_each_niche.pdf"))

In [ ]:
latent_cluster_key = 'niche_type'
cell_type_key = 'transferred_cell_type_unfiltered'

In [ ]:
adata_query

#### 保存 model 结果

In [ ]:
# Save trained model
model_folder_path = f"{workdir}/model_stereo_slide"
os.makedirs(model_folder_path, exist_ok=True)

new_model.save(dir_path=model_folder_path,
           overwrite=True,
           save_adata=True,
           adata_file_name="adata_concat_stereo_slide.h5ad")

In [ ]:
from Garfield.model import Garfield

workdir = f'/pri_exthome/zhouwg/project/Garfield_benchmark/results/sp_unimodal/spRNA_integrated'
gf.settings.set_workdir(workdir)
model_folder_path = f"{workdir}/model_stereo_slide"

new_model = Garfield.load(dir_path=model_folder_path,
                      adata_file_name="adata_concat_stereo_slide.h5ad")

In [ ]:
adata_query

In [ ]:
### 输出celltype 和 空间位置数据供R绘图
# Extracting 'latent_leiden_0.3' and 'spatial' data
# cell_type_key = 'transferred_niche_type_unfiltered'
# latent_leiden_data = adata_query.obs[cell_type_key]
niche_data = adata_query.obs['transferred_niche_type_unfiltered']
spatial_data = adata_query.obsm["spatial"]

# Creating a DataFrame for R visualization
output_df = pd.DataFrame({
    # cell_type_key: latent_leiden_data,
    'transferred_niche_type_unfiltered': niche_data,
    "spatial_X": spatial_data[:, 0],
    "spatial_Y": spatial_data[:, 1],
})

# Save to CSV
output_file = f"latent_transfer_niche_type_data_{dataset_name}.csv"
output_df.to_csv(os.path.join(workdir, output_file))

In [ ]:
### 输出celltype 和 空间位置数据供R绘图
# Extracting 'latent_leiden_0.3' and 'spatial' data
# cell_type_key = 'transferred_niche_type_unfiltered'
# latent_leiden_data = adata_query.obs[cell_type_key]
niche_data = adata_query.obs['transferred_niche_type_unfiltered']
spatial_data = adata_query.obsm["X_umap"]

# Creating a DataFrame for R visualization
output_df = pd.DataFrame({
    # cell_type_key: latent_leiden_data,
    'transferred_niche_type_unfiltered': niche_data,
    "umap_1": spatial_data[:, 0],
    "umap_2": spatial_data[:, 1],
})

# Save to CSV
output_file = f"latent_transfer_umap_data_{dataset_name}.csv"
output_df.to_csv(os.path.join(workdir, output_file))

In [ ]:
adata_query

In [ ]:
dataset_name

In [ ]:
### 绘制桑基图
import matplotlib.pyplot as plt
import numpy as np
import scanpy as sc
from matplotlib import gridspec
import collections
import colorsys
import matplotlib

def get_distinct_colors(n):
    """
    https://www.quora.com/How-do-I-generate-n-visually-distinct-RGB-colours-in-Python/answer/Karthik-Kumar-Viswanathan
    """
    hue_partition = 1 / (n + 1)
    colors = [
        colorsys.hsv_to_rgb(hue_partition * value, 1.0, 1.0) for value in range(0, n)
    ]
    return colors[::2] + colors[1::2]


def text_width(fig, ax, text, fontsize):
    text = ax.text(-100, 0, text, fontsize=fontsize)
    text_bb = text.get_window_extent(renderer=fig.canvas.get_renderer())
    text_bb = text_bb.transformed(fig.dpi_scale_trans.inverted())
    width = text_bb.width
    text.remove()
    return width

class Sankey:
    def __init__(
        self,
        x,
        y,
        colorside,
        plot_width=8,
        plot_height=8,
        gap=0.12,
        alpha=0.3,
        fontsize="small",
        left_order=None,
        mapping=None,
        colors=None,
        #                  colorside=None,
        tag=None,
        title=None,
        title_left=None,
        title_right=None,
        ax=None,
    ):
        self.X = x
        self.Y = y
        if ax:
            self.plot_width = ax.get_position().width * ax.figure.get_size_inches()[0]
            self.plot_height = ax.get_position().height * ax.figure.get_size_inches()[1]
        else:
            self.plot_width = plot_width
            self.plot_height = plot_height
        self.gap = gap
        self.alpha = alpha
        self.colors = colors
        self.colorside = colorside
        self.fontsize = fontsize
        self.tag = tag
        self.map = mapping is not None
        self.mapping = mapping
        self.mapping_colors = {
            "increase": "#1f721c",
            "decrease": "#ddc90f",
            "mistake": "#dd1616",
            "correct": "#dddddd",
            "novel": "#59a8d6",
        }
        self.title = title
        self.title_left = title_left
        self.title_right = title_right

        self.need_title = any(
            map(lambda x: x is not None, (title, title_left, title_right))
        )
        if self.need_title:
            self.plot_height -= 0.5

        self.init_figure(ax)

        self.flows = collections.Counter(zip(x, y))
        self.init_nodes(left_order)

        self.init_widths()
        # inches per 1 item in x and y
        self.resolution = (plot_height - gap * (len(self.left_nodes) - 1)) / len(x)
        if self.colors == None:
            if colorside == "left":
                self.colors = {
                    name: colour
                    for name, colour in zip(
                        self.left_nodes.keys(),
                        get_distinct_colors(len(self.left_nodes)),
                    )
                }
            elif colorside == "right":
                self.colors = {
                    name: colour
                    for name, colour in zip(
                        self.right_nodes.keys(),
                        get_distinct_colors(len(self.right_nodes)),
                    )
                }
            else:
                raise ValueError(
                    "colorside argument should be set either to 'left' or 'right'. Exiting."
                )

        self.init_offsets()

    def init_figure(self, ax):
        if ax is None:
            self.fig = plt.figure()
            self.ax = plt.Axes(self.fig, [0, 0, 1, 1])
            self.fig.add_axes(self.ax)
        self.fig = ax.figure
        self.ax = ax

    def init_nodes(self, left_order):
        left_nodes = {}
        right_nodes = {}
        left_offset = 0
        for (left, right), flow in self.flows.items():
            if left in left_nodes:
                left_nodes[left] += flow
            else:
                left_nodes[left] = flow
            if right in right_nodes:
                node = right_nodes[right]
                node[0] += flow
                if flow > node[2]:
                    node[1] = left
                    node[2] = flow
            else:
                right_nodes[right] = [flow, left, flow]

        self.left_nodes = collections.OrderedDict()
        self.left_nodes_idx = {}
        if left_order is None:
            key = lambda pair: -pair[1]
        else:
            left_order = list(left_order)
            key = lambda pair: left_order.index(pair[0])

        for name, flow in sorted(left_nodes.items(), key=key):
            self.left_nodes[name] = flow
            self.left_nodes_idx[name] = len(self.left_nodes_idx)

        left_names = list(self.left_nodes.keys())
        self.right_nodes = collections.OrderedDict()
        self.right_nodes_idx = {}
        for name, node in sorted(
            right_nodes.items(),
            key=lambda pair: (left_names.index(pair[1][1]), -pair[1][2]),
        ):
            self.right_nodes[name] = node[0]
            self.right_nodes_idx[name] = len(self.right_nodes_idx)

    def init_widths(self):
        self.left_width = max(
            (
                text_width(self.fig, self.ax, node, self.fontsize)
                for node in self.left_nodes
            )
        )
        if self.title_left:
            self.left_width = max(
                self.left_width,
                text_width(self.fig, self.ax, self.title_left, self.fontsize) / 2,
            )
        self.right_width = max(
            (
                text_width(self.fig, self.ax, node, self.fontsize)
                for node in self.right_nodes
            )
        )
        if self.title_right:
            self.right_width = max(
                self.right_width,
                text_width(self.fig, self.ax, self.title_right, self.fontsize) / 2,
            )

        self.right_stop = self.plot_width - self.left_width - self.right_width
        self.middle1_stop = self.right_stop * 9 / 20
        self.middle2_stop = self.right_stop * 11 / 20

    def init_offsets(self):
        self.offsets_l = {}
        self.offsets_r = {}

        offset = 0
        for name, flow in self.left_nodes.items():
            self.offsets_l[name] = offset
            offset += flow * self.resolution + self.gap

        offset = 0
        for name, flow in self.right_nodes.items():
            self.offsets_r[name] = offset
            offset += flow * self.resolution + self.gap

    def draw_flow(self, left, right, flow, node_offsets_l, node_offsets_r, colorside):
        P = matplotlib.path.Path

        flow *= self.resolution
        left_y = self.offsets_l[left] + node_offsets_l[left]
        right_y = self.offsets_r[right] + node_offsets_r[right]
        if self.need_title:
            left_y += 0.5
            right_y += 0.5
        node_offsets_l[left] += flow
        node_offsets_r[right] += flow
        if colorside == "left":
            color = self.colors[left]
        elif colorside == "right":
            color = self.colors[right]
        if self.mapping is not None:
            color = self.mapping_colors[self.mapping.category(left, right)]

        path_data = [
            (P.MOVETO, (0, -left_y)),
            (P.LINETO, (0, -left_y - flow)),
            (P.CURVE4, (self.middle1_stop, -left_y - flow)),
            (P.CURVE4, (self.middle2_stop, -right_y - flow)),
            (P.CURVE4, (self.right_stop, -right_y - flow)),
            (P.LINETO, (self.right_stop, -right_y)),
            (P.CURVE4, (self.middle2_stop, -right_y)),
            (P.CURVE4, (self.middle1_stop, -left_y)),
            (P.CURVE4, (0, -left_y)),
            (P.CLOSEPOLY, (0, -left_y)),
        ]
        codes, verts = zip(*path_data)
        path = P(verts, codes)
        patch = matplotlib.patches.PathPatch(
            path,
            facecolor=color,
            alpha=0.9 if flow < 0.02 else self.alpha,
            edgecolor="none",
        )
        self.ax.add_patch(patch)

    def draw_label(self, label, is_left):
        nodes = self.left_nodes if is_left else self.right_nodes
        offsets = self.offsets_l if is_left else self.offsets_r
        y = offsets[label] + nodes[label] * self.resolution / 2
        if self.need_title:
            y += 0.5

        self.ax.text(
            -0.1 if is_left else self.right_stop + 0.1,
            -y,
            label,
            horizontalalignment="right" if is_left else "left",
            verticalalignment="center",
            fontsize=self.fontsize,
        )

    def draw_titles(self):
        if self.title:
            self.ax.text(
                self.right_stop / 2,
                -0.25,
                self.title,
                horizontalalignment="center",
                verticalalignment="center",
                fontsize=self.fontsize,
                fontweight="bold",
            )
        if self.title_left:
            self.ax.text(
                -0.1,
                -0.25,
                self.title_left,
                horizontalalignment="right",
                verticalalignment="center",
                fontsize=self.fontsize,
            )
        if self.title_right:
            self.ax.text(
                self.right_stop + 0.1,
                -0.25,
                self.title_right,
                horizontalalignment="left",
                verticalalignment="center",
                fontsize=self.fontsize,
            )

    def draw(self, colorside):
        node_offsets_l = collections.Counter()
        node_offsets_r = collections.Counter()

        for (left, right), flow in sorted(
            self.flows.items(),
            key=lambda pair: (
                self.left_nodes_idx[pair[0][0]],
                self.right_nodes_idx[pair[0][1]],
            ),
        ):
            self.draw_flow(left, right, flow, node_offsets_l, node_offsets_r, colorside)

        for name in self.left_nodes:
            self.draw_label(name, True)
        for name in self.right_nodes:
            self.draw_label(name, False)
        self.draw_titles()

        self.ax.axis("equal")
        self.ax.set_xlim(
            -self.left_width - self.gap, self.right_stop + self.gap + self.right_width
        )
        self.ax.get_xaxis().set_visible(False)
        self.ax.get_yaxis().set_visible(False)
        for k in self.ax.spines.keys():
            self.ax.spines[k].set_visible(False)
        # plt.axis('off')
        # self.fig.set_figheight(self.plot_height)
        # self.fig.set_figwidth(self.plot_width)
        if self.tag:
            text_ax = self.fig.add_axes((0.02, 0.95, 0.05, 0.05), frame_on=False)
            text_ax.set_axis_off()
            plt.text(
                0, 0, self.tag, fontsize=30, transform=text_ax.transAxes
            )
        # plt.tight_layout()


def sankey(x, y, colorside="left", **kwargs):
    diag = Sankey(x, y, colorside, **kwargs)
    diag.draw(colorside)
    return diag.fig

In [ ]:
tmp = sc.read_h5ad('/pri_exthome/zhouwg/project/Garfield_benchmark/results/sp_unimodal/spRNA_integrated/model_slide/adata_slide.h5ad')
adata_query.obs['niche_type'] = tmp.obs['niche_type']
adata_query

In [ ]:
list(adata_query.obs['niche_type'].unique())

In [ ]:
tmp.obs['transferred_niche_type_unfiltered'].unique()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# 假设 x 和 y 是字符型数据
# tmp = adata_query[adata_query.obs['niche_type'].isin(['ONL', 'GL', 'EPL_inner', 'EPL_outer',
#                                                       'GCL_external', 'GCL_deep']), :].copy()
# tmp = tmp[tmp.obs['transferred_niche_type_unfiltered'].isin(['ONL', 'GL',
#                                                              'EPL', 'GCL_external', 'GCL_deep']), :].copy()
x = adata_query.obs['niche_type']
y = adata_query.obs['transferred_niche_type_unfiltered']

# 创建一个字符型的映射
unique_x = pd.Series(x).unique()
unique_y = pd.Series(y).unique()

# 为每个类别分配一个独特的索引
x_mapping = {label: idx for idx, label in enumerate(unique_x)}
y_mapping = {label: idx for idx, label in enumerate(unique_y)}

# 将 x 和 y 数据转换为它们对应的索引
x_numeric = np.array([x_mapping[label] for label in x])
y_numeric = np.array([y_mapping[label] for label in y])

# 定义自定义的颜色
custom_colors = ["#F3BFCBFF", "#EB5291FF", "#FBBB68FF", "#C3EF00FF", "#9DDAF5FF", "#6351A0FF",
                 "#E16E6E", "#FEF79EFF", "#1794CEFF", "#972C8DFF", "#026CCBFF"]

# 创建一个图表和轴
fig, ax = plt.subplots(figsize=(10, 8))

# 创建 Sankey 实例并绘制
sankey = Sankey(x=x_numeric, y=y_numeric, colorside='left',
                colors=custom_colors, ax=ax,
                plot_width=10, plot_height=8, gap=0.15, alpha=0.5, fontsize="medium",
                title="Label transfer (Slide-seqV2_MoB)")

sankey.draw(colorside='left')
plt.show()

In [ ]:
x_mapping

In [ ]:
y_mapping

In [ ]:
# 设置数据流动的 x, y 坐标
x = adata_query.obs['transferred_niche_type_unfiltered'].values
y = adata_query.obs['niche_type'].values

# 绘制图表
fig, ax = plt.subplots(figsize=(10, 8))
# 创建一个 Sankey 实例
custom_colors = ["#F3BFCBFF", "#EB5291FF", "#FBBB68FF", "#C3EF00FF", "#9DDAF5FF", "#6351A0FF",
                 "#ECF1F4FF", "#FEF79EFF", "#1794CEFF", "#972C8DFF", "#026CCBFF"]
sankey = Sankey(x=x, y=y, colorside='left',
                colors=custom_colors,
                ax=ax, plot_width=10,
                plot_height=8, gap=0.15, alpha=0.5, fontsize="medium",
                title="Label transfer (Slide-seqV2_MoB)")
sankey.draw(colorside='left')
plt.show()

### slideseq数据单独 niches 分析

#### Integrating spatially resolved transcriptomics data using Garfield

In [ ]:
# set workdir #
workdir = f'/pri_exthome/zhouwg/project/Garfield_benchmark/results/sp_unimodal/spRNA_integrated'
gf.settings.set_workdir(workdir)

### modify parameter
user_config = dict(
    ## Input options
    adata_list=adata_slide,
    profile='spatial',
    data_type='single-modal',
    sample_col='batch', # batch
    weight=0.5,
    ## Preprocessing options
    graph_const_method='mu_std', # mu_std, Radius, KNN, Squidpy
    used_hvg=True,
    min_cells=3,
    min_features=0,
    keep_mt=False,
    target_sum=1e4,
    rna_n_top_features=3000,
    n_components=50,
    n_neighbors=5,
    metric='euclidean',
    svd_solver='arpack',
    # datasets
    used_pca_feat=False,
    adj_key='connectivities',
    # data split parameters
    edge_val_ratio=0.1,
    edge_test_ratio=0.,
    node_val_ratio=0.1,
    node_test_ratio=0.,
    ## Model options
    augment_type='svd', # svd
    svd_q=5,
    use_FCencoder=True,
    conv_type='GATv2Conv', # GAT or GATv2Conv or GCN
    gnn_layer=2,
    hidden_dims=[128, 128],
    bottle_neck_neurons=20,
    cluster_num=20,
    drop_feature_rate=0.2,
    drop_edge_rate=0.2,
    num_heads=3,
    dropout=0.2,
    concat=True,
    used_edge_weight=True,
    used_DSBN=False,
    used_mmd=False,
    # data loader parameters
    num_neighbors=5,
    loaders_n_hops=2,
    edge_batch_size=4096,
    node_batch_size=512, # None
    # loss parameters
    include_edge_recon_loss=True,
    include_gene_expr_recon_loss=True,
    lambda_latent_contrastive_instanceloss=1.0,
    lambda_latent_contrastive_clusterloss=0.5,
    lambda_gene_expr_recon=1., #
    lambda_edge_recon=10., #
    lambda_latent_adj_recon_loss=2.,
    lambda_omics_recon_mmd_loss=0.5,
    # train parameters
    n_epochs_no_edge_recon=0,
    learning_rate=0.001,
    weight_decay=1e-05,
    gradient_clipping=5,
    # other parameters
    latent_key='garfield_latent',
    reload_best_model=True,
    use_early_stopping=True,
    early_stopping_kwargs=None,
    monitor=True,
    device_id=0,
    seed=2024,
    verbose=True
)
dict_config = gf.settings.set_gf_params(user_config)

In [ ]:
from Garfield.model import Garfield

# Initialize model
model = Garfield(dict_config)

In [ ]:
# Train model
model.train()

In [ ]:
# Compute latent neighbor graph
latent_key = 'garfield_latent'
sc.pp.neighbors(model.adata,
                use_rep=latent_key,
                key_added=latent_key)
# Compute UMAP embedding
sc.tl.umap(model.adata,
           neighbors_key=latent_key)

In [ ]:
# Compute latent Leiden clustering
latent_leiden_resolution = 0.5
latent_cluster_key = f"latent_leiden_{str(latent_leiden_resolution)}"
latent_key = "garfield_latent"
cell_type_key = 'cell_type'

# louvain leiden
sc.tl.leiden(adata=model.adata,
             resolution=latent_leiden_resolution,
             key_added=latent_cluster_key,
             neighbors_key=latent_key)
len(model.adata.obs[latent_cluster_key].unique())

#### Visualize Garfield Latent Space

In [ ]:
sc.settings.set_figure_params(dpi=100, facecolor='white')

sc.pl.umap(model.adata, color=['batch', latent_cluster_key],
           s=10, show=False, ncols=2, wspace=0.5) # , legend_loc='on data'

In [ ]:
## 尝试亚群重细分，效果一般
tmp = model.adata.copy()
# 使用 restrict_to 限制聚类范围
# leiden
sc.tl.louvain(tmp, restrict_to=(latent_cluster_key, ['2']), resolution = 0.3)
tmp.obs['louvain_R'].value_counts()
import pandas as pd

# Step 1: Count cells in each leiden_R group
leiden_counts = tmp.obs['louvain_R'].value_counts()

# Step 2: Identify small clusters (< 50 cells)
small_clusters = leiden_counts[leiden_counts < 50].index

# Step 3: Merge small clusters back to original latent_cluster_key
def merge_small_clusters(row):
    if row['louvain_R'] in small_clusters:
        return row[latent_cluster_key]  # Use original cluster name
    return row['louvain_R']  # Keep new cluster if size >= 50

tmp.obs['merged_cluster'] = tmp.obs.apply(merge_small_clusters, axis=1)

# Step 4: Assign new labels for large clusters (>= 50 cells) starting from 0
large_clusters = tmp.obs['merged_cluster'].unique()
new_label_mapping = {label: i for i, label in enumerate(sorted(large_clusters))}
tmp.obs['final_cluster'] = tmp.obs['merged_cluster'].map(new_label_mapping)
tmp.obs['final_cluster'] = tmp.obs['final_cluster'].astype('category')

# Step 5: Visualize results
sc.pl.umap(
    tmp,
    color=[latent_cluster_key, 'louvain_R', 'final_cluster'],
    title=['Original Cluster', 'Reclustered', 'Final Cluster']
)


In [ ]:
import re

from matplotlib import pyplot as plt
import matplotlib
matplotlib.use("Agg") #使用非交互式的后端生成图像文件

def pic(pdf):
    searchObj = re.search( r'(.*).pdf', pdf)
    png = f"{searchObj.group(1)}.png"
    plt.savefig(pdf, bbox_inches="tight")
    plt.savefig(png, bbox_inches="tight", dpi=300)
    plt.close()

In [ ]:
import matplotlib.pyplot as plt
model.adata.obsm['spatial'][:, 1] *= -1  # 翻转 y 坐标 手动修正

In [ ]:
sc.pl.embedding(model.adata,
                basis='spatial', color=latent_cluster_key,
                title='Slide-seqV2', s=20, show=False,
                frameon=False)
# plt.gca().invert_yaxis()  # 翻转 y 轴

In [ ]:
## 提取特定的batch数据集
import math
import matplotlib.pyplot as plt

batch = 'Slide-seqV2'
adata_batch = model.adata#[model.adata.obs['batch'] == batch].copy()
sc.pl.umap(adata_batch, color=[latent_cluster_key],
           s=10, show=False, ncols=2, wspace=0.3)

cluster_list = list(adata_batch.obs[latent_cluster_key].unique()) # ['0', '1', '3']
# cluster_list = ['7', '3', '12', '2',
#                 '6', '0', '16', '10']
highlight_color = adata_batch.uns[f'{latent_cluster_key}_colors']
default_color = 'lightgray'  # 为其他细胞设置灰色
fig, ax_list = plt.subplots(
    2, math.ceil(len(cluster_list) / 2), figsize=(2 * len(cluster_list), 8)
)
ax_list = ax_list.flatten()  # 展平以便在循环中逐一使用

tmp = adata_batch.copy()
for cluster, ax in zip(cluster_list, ax_list):
    # 设置所有细胞颜色为灰色
    tmp.obs['temp_color'] = default_color
    # 将特定 cluster 设置为高亮颜色
    cluster_cells = tmp.obs[latent_cluster_key] == cluster
    tmp.obs.loc[cluster_cells, 'temp_color'] = highlight_color[int(cluster)]

    # 绘制空间图
    sc.pl.embedding(
        tmp,
        basis='spatial',
        color='temp_color',
        palette=[highlight_color[int(cluster)], default_color],
        ax=ax,
        show=False,
        s=20,
        title=f"Niche {cluster}",
        legend_loc=None
    )

plt.tight_layout(w_pad=0.3)
# plt.show()
# pic(os.path.join(workdir, "03.spatial_plot_each_niche.pdf"))

#### 找 niche 差异基因

In [ ]:
import scanpy as sc
sc.pp.normalize_total(adata_slide, target_sum=1e4)
sc.pp.log1p(adata_slide)

In [ ]:
latent_cluster_key = f"latent_leiden_{str(latent_leiden_resolution)}"

adata_slide.obs[latent_cluster_key] = model.adata.obs[latent_cluster_key]
sc.tl.rank_genes_groups(adata_slide, latent_cluster_key, method='wilcoxon')

In [ ]:
def extract_top_genes_spatial(
    adata,
    workdir,
    latent_cluster_key,
    n_top_markers=5,
    diy_marker_list=None,
    output_file="all_marker_genes_spatial_data.csv",
    use_normalized=False,
    remove_mitochondrial=True,
    remove_red_blood_cells=True,
    species="human"  # species can be 'human' or 'mouse'
):
    """
    提取 AnnData 对象中所有 marker 基因的表达数据、空间位置信息和 cluster 信息，并保存为 CSV 文件。
    移除线粒体基因和红细胞相关基因。适配人类或小鼠数据。

    Parameters:
    -----------
    adata : AnnData
        包含 rank_genes_groups 和空间信息的 AnnData 对象。
    workdir : str
        输出文件的保存目录。
    latent_cluster_key : str
        cluster 信息所在的 obs 键。
    output_file : str
        保存的 CSV 文件名。
    use_normalized : bool
        如果为 True，使用标准化后的数据 (.X)；否则使用 counts 层数据。
    remove_mitochondrial : bool
        如果为 True，移除线粒体基因。
    remove_red_blood_cells : bool
        如果为 True，移除红细胞相关基因。
    species : str
        数据来源物种，"human" 或 "mouse"。

    Returns:
    --------
    None
    """
    import os
    import pandas as pd

    # 检查 rank_genes_groups 是否存在
    if "rank_genes_groups" not in adata.uns:
        raise ValueError("The AnnData object does not contain 'rank_genes_groups' in .uns.")

    # 提取所有 marker 基因
    ranked_genes = adata.uns['rank_genes_groups']
    marker_genes = set()  # 使用集合避免重复
    clusters = ranked_genes['names'].dtype.names  # 获取所有 cluster 名

    # 定义线粒体基因和红细胞基因的过滤列表
    mitochondrial_prefix = 'MT-' if species == "human" else 'mt-'
    red_blood_cell_genes = {
        "human": ["HBA1", "HBA2", "HBB", "HBG1", "HBG2", "HBD", "HBQ1", "HBS1", "RBC"],
        "mouse": ["Hba-a1", "Hba-a2", "Hbb-b1", "Hbb-b2", "Hbd", "Hbb", "Rbc"]
    }

    # 收集所有需要排除的基因
    excluded_genes = set()
    if remove_mitochondrial:
        excluded_genes.update({gene for gene in adata.var_names if gene.startswith(mitochondrial_prefix)})
    if remove_red_blood_cells:
        excluded_genes.update(red_blood_cell_genes.get(species, []))

    # 提取 marker 基因并排除不需要的基因
    cluster_marker_genes = {}  # 用来存储每个cluster的最终marker基因
    for cluster in clusters:
        top_genes = ranked_genes['names'][cluster][:n_top_markers]  # 获取Top n 基因
        current_marker_genes = set(top_genes)  # 当前 cluster 的 marker 基因

        # 排除不需要的基因
        current_marker_genes -= excluded_genes

        # 更新最终的marker基因集合
        cluster_marker_genes[cluster] = list(current_marker_genes)

        # 打印每个cluster的最终marker基因
        print(f"Cluster {cluster}: {list(current_marker_genes)}")

    # 汇总所有 cluster 的 marker 基因
    marker_genes = set([gene for genes in cluster_marker_genes.values() for gene in genes])

    # 提示输出
    print(f"Extracted {len(marker_genes)} total unique marker genes after filtering.")

    # 确保所有marker基因都在adata.var_names中
    marker_genes = [gene for gene in marker_genes if gene in adata.var_names]
    if len(marker_genes) == 0:
        raise ValueError("No valid marker genes found in the AnnData object.")

    ## 拓展现有的marker，增加人为定义的
    if diy_marker_list is not None:
        marker_genes.extend(diy_marker_list)

    # 提取表达数据
    if use_normalized:
        gene_expression = adata[:, marker_genes].X  # 使用标准化数据
    else:
        gene_expression = adata[:, marker_genes].layers['counts']  # counts 数据

    # 提取空间位置信息
    if 'spatial' not in adata.obsm:
        raise ValueError("Spatial information not found in adata.obsm.")
    spatial_positions = adata.obsm['spatial']

    # 提取 cluster 信息
    clusters = adata.obs[latent_cluster_key]

    # 构建最终 DataFrame
    final_dataframe = pd.DataFrame(
        gene_expression.toarray() if hasattr(gene_expression, "toarray") else gene_expression,
        columns=marker_genes,
        index=adata.obs.index
    )

    # 添加空间位置信息
    final_dataframe['x'] = spatial_positions[:, 0]  # 添加 x 坐标
    final_dataframe['y'] = spatial_positions[:, 1]  # 添加 y 坐标

    # 添加 cluster 信息
    final_dataframe['cluster'] = clusters

    # 输出最终的 CSV 文件
    final_dataframe.to_csv(os.path.join(workdir, output_file))

    print(f"Marker genes data saved to {os.path.join(workdir, output_file)}.")

In [ ]:
df  = sc.get.rank_genes_groups_df(adata_slide, group=None)
df = df.sort_values(by="scores", ascending=False)
df.to_csv(os.path.join(workdir, 'diff_niches_slide.csv'), index=0, sep='\t')

In [ ]:
# 给定一个基因，判断是否在adata.var_names里
'Slc6a11' in adata_slide.var_names

In [ ]:
## output res
extract_top_genes_spatial(
    adata_slide,
    workdir,
    latent_cluster_key,
    diy_marker_list = ['Slc6a11', 'Vip'],
    n_top_markers=10,
    output_file="top_genes_spatial_data_Slide-seqV2_MoB.csv",
    use_normalized=True,
    remove_mitochondrial=True,
    remove_red_blood_cells=True,
    species='mouse'
)

In [ ]:
## niche 注释
cluster2annotation = {
    '0': 'GCL_external', # Pcp4
    '1': 'EPL_inner', # Cst3
    '2': 'GCL&RMS', # Malat1&Mbp
    '3': 'MCL', # Vip Slc25a4
    '4': 'ONL', # S100a5
    '5': 'EPL_outer', # Socs3
    '6': 'IPL', # Slc17a7
    '7': 'RMS', # Mbp
    '8': 'GCL_deep', # Nrxn3
    '9': 'ONL', # S100a5
    '10': 'Meninge', # Ptgds
    '11': 'GL', # Calb2
    '12': 'GCL_deep'
}
model.adata.obs['niche_type'] = model.adata.obs[latent_cluster_key].map(cluster2annotation).astype('category')

In [ ]:
sc.pl.umap(model.adata, color=['niche_type'],
           s=10, show=False, ncols=2, wspace=0.3, legend_loc='on data')

In [ ]:
sc.pl.umap(model.adata, color=[latent_cluster_key],
           s=10, show=False, ncols=2, wspace=0.3, legend_loc='on data')

#### 保存 model 结果

In [ ]:
# Save trained model
model_folder_path = f"{workdir}/model_slide"
os.makedirs(model_folder_path, exist_ok=True)

model.save(dir_path=model_folder_path,
           overwrite=True,
           save_adata=True,
           adata_file_name="adata_slide.h5ad")

In [ ]:
from Garfield.model import Garfield

workdir = f'/pri_exthome/zhouwg/project/Garfield_benchmark/results/sp_unimodal/spRNA_integrated'
gf.settings.set_workdir(workdir)
model_folder_path = f"{workdir}/model_slide"

model = Garfield.load(dir_path=model_folder_path,
              adata_file_name="adata_slide.h5ad")

In [ ]:
import pandas as pd

# Define a function to compute CHAOS score for a range of resolutions
def compute_chaos_scores(adata, resolutions, latent_key="garfield_latent"):
    results = []
    for resolution in resolutions:
        latent_cluster_key = f"latent_leiden_{resolution}"

        # Perform Leiden clustering
        sc.tl.leiden(adata=adata,
                      resolution=resolution,
                      key_added=latent_cluster_key,
                      neighbors_key=latent_key)

        # Compute CHAOS score
        X = adata.obsm['spatial']
        pred_labels = adata.obs[latent_cluster_key].values
        chaos = CHAOS_score(X=X, pred_labels=pred_labels)

        # Store results
        results.append({"Resolution": resolution, "N_niches": len(model.adata.obs[latent_cluster_key].unique()),"CHAOS_Score": chaos})

    # Convert results to a DataFrame
    df_results = pd.DataFrame(results)
    return df_results

# Define a list of resolutions 0.5 highlight
resolutions = [0.3, 0.4, 0.5, 0.6, 0.7]

# Call the function with your adata object and resolutions
# Assuming `model.adata` is the AnnData object used
chaos_results_df = compute_chaos_scores(adata=model.adata, resolutions=resolutions)

# Save the results for R visualization
chaos_results_df.to_csv(os.path.join(workdir, "chaos_results_slide.csv"), index=False)

In [ ]:
chaos_results_df

In [ ]:
### 输出niche 和 空间位置数据供R绘图
# Extracting 'latent_leiden_0.3' and 'spatial' data
dataset_name = 'Slide-seqV2_MoB'
visua_type = 'niche_type'
latent_leiden_data = model.adata.obs[visua_type] # latent_cluster_key
spatial_data = model.adata.obsm["spatial"]

# Creating a DataFrame for R visualization
output_df = pd.DataFrame({
    visua_type: latent_leiden_data,
    "spatial_X": spatial_data[:, 0],
    "spatial_Y": spatial_data[:, 1],
})

# Save to CSV
output_file = f"latent_spatial_data_{dataset_name}.csv"
output_df.to_csv(os.path.join(workdir, output_file))

In [ ]:
### 输出niche 和 UMAP位置数据供R绘图
# Extracting 'latent_leiden_0.3' and 'spatial' data
visua_type = 'niche_type'
latent_leiden_data = model.adata.obs[visua_type] # latent_cluster_key
spatial_data = model.adata.obsm["X_umap"]

# Creating a DataFrame for R visualization
output_df = pd.DataFrame({
    visua_type: latent_leiden_data,
    "umap_1": spatial_data[:, 0],
    "umap_2": spatial_data[:, 1],
})

# Save to CSV
output_file = f"latent_umap_data_{dataset_name}.csv"
output_df.to_csv(os.path.join(workdir, output_file))

In [ ]:
dataset_name

### 结合单细胞数据集，看niche的celltype分布

In [ ]:
import scanpy as sc
import pandas as pd

def read_expr_and_meta(expression_file, metadata_file):
    # 读取基因表达矩阵 (行为基因，列为细胞)
    expression_data = pd.read_csv(expression_file, index_col=0)
    # 读取metadata信息 (列为细胞，行是样本或细胞类型)
    metadata = pd.read_csv(metadata_file, index_col=0)

    # 找到细胞名称的交集
    common_cells = expression_data.columns.intersection(metadata.index)

    if len(common_cells) == 0:
        raise ValueError("没有共同的细胞！请检查表达矩阵和元数据文件。")

    # 对表达矩阵和metadata进行子集化，只保留共同的细胞
    expression_data = expression_data[common_cells]
    metadata = metadata.loc[common_cells]

    # 创建scanpy的AnnData对象
    adata = sc.AnnData(X=expression_data.values.T)  # 转置，行是细胞，列是基因
    adata.obs = metadata  # 将metadata赋值给adata.obs
    adata.var = pd.DataFrame(index=expression_data.index)  # 将基因名称赋给adata.var

    return adata

# 使用示例
root_dir = '/pri_exthome/zhouwg/project/spatial_data/mouse_olfactory_bulb'
expression_file = 'GSE121891_OB_6_runs.raw.dge.csv'
metadata_file = 'GSE121891_OB_metaData_seurat.csv'
adata_ref = read_expr_and_meta(os.path.join(root_dir, expression_file),
                           os.path.join(root_dir, metadata_file))
adata_ref

In [ ]:
adata_ref.obs['ClusterName'].value_counts()

In [ ]:
adata_ref.obs['orig.ident'].value_counts()

In [ ]:
## 快速看看是否存在 batch
adata = adata_ref.copy()
sc.pp.filter_cells(adata, min_genes=100)
sc.pp.filter_genes(adata, min_cells=3)

# Saving count data
adata.layers["counts"] = adata.X.copy()
# Normalizing to median total counts
sc.pp.normalize_total(adata)
# Logarithmize the data:
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, n_top_genes=2000)
sc.tl.pca(adata)
sc.pp.neighbors(adata)
sc.tl.umap(adata)
sc.pl.umap(adata, color=["orig.ident", "ClusterName"], wspace=0.3)

In [ ]:
## 不存在明显 batch 不做批次矫正
del adata

#### Integrating single-cell resolved transcriptomics data using Garfield

In [ ]:
# set workdir #
workdir = f'/pri_exthome/zhouwg/project/Garfield_benchmark/results/sp_unimodal/spRNA_integrated'
gf.settings.set_workdir(workdir)

### modify parameter
user_config = dict(
    ## Input options
    adata_list=adata_ref,
    profile='RNA',
    data_type='single-modal',
    sample_col=None,
    weight=0.5,
    ## Preprocessing options
    graph_const_method='mu_std', # mu_std, Radius, KNN, Squidpy
    used_hvg=True,
    min_cells=3,
    min_features=0,
    keep_mt=False,
    target_sum=1e4,
    rna_n_top_features=3000,
    n_components=50,
    n_neighbors=5,
    metric='euclidean',
    svd_solver='arpack',
    # datasets
    used_pca_feat=False,
    adj_key='connectivities',
    # data split parameters
    edge_val_ratio=0.1,
    edge_test_ratio=0.,
    node_val_ratio=0.1,
    node_test_ratio=0.,
    ## Model options
    augment_type='svd',
    svd_q=5,
    use_FCencoder=False,
    conv_type='GATv2Conv', # GAT or GATv2Conv or GCN
    gnn_layer=2,
    hidden_dims=[128, 128],
    bottle_neck_neurons=20,
    cluster_num=20,
    drop_feature_rate=0.2,
    drop_edge_rate=0.2,
    num_heads=3,
    dropout=0.2,
    concat=True,
    used_edge_weight=False,
    used_DSBN=False,
    used_mmd=True,
    # data loader parameters
    num_neighbors=5,
    loaders_n_hops=2,
    edge_batch_size=4096,
    node_batch_size=256, # None
    # loss parameters
    include_edge_recon_loss=True,
    include_gene_expr_recon_loss=True,
    lambda_latent_contrastive_instanceloss=1.0,
    lambda_latent_contrastive_clusterloss=0.5,
    lambda_gene_expr_recon=5., #
    lambda_edge_recon=1., #
    lambda_latent_adj_recon_loss=2.,
    lambda_omics_recon_mmd_loss=1.0,
    # train parameters
    n_epochs_no_edge_recon=0,
    learning_rate=0.001,
    weight_decay=1e-05,
    gradient_clipping=5,
    # other parameters
    latent_key='garfield_latent',
    reload_best_model=True,
    use_early_stopping=True,
    early_stopping_kwargs=None,
    monitor=True,
    device_id=0,
    seed=2024,
    verbose=True
)
dict_config = gf.settings.set_gf_params(user_config)

In [ ]:
from Garfield.model import Garfield

# Initialize model
model = Garfield(dict_config)

In [ ]:
# Train model
model.train()

In [ ]:
# Compute latent neighbor graph
latent_key = 'garfield_latent'
sc.pp.neighbors(model.adata,
                use_rep=latent_key,
                key_added=latent_key)
# Compute UMAP embedding
sc.tl.umap(model.adata,
           neighbors_key=latent_key)

In [ ]:
model.adata.obs['cell_type'] = model.adata.obs['ClusterName']

In [ ]:
# Compute latent Leiden clustering
latent_leiden_resolution = 0.8
latent_cluster_key = f"latent_leiden_{str(latent_leiden_resolution)}"
latent_key = "garfield_latent"
cell_type_key = 'cell_type'

sc.tl.leiden(adata=model.adata,
             resolution=latent_leiden_resolution,
             key_added=latent_cluster_key,
             neighbors_key=latent_key)
len(model.adata.obs[latent_cluster_key].unique())

In [ ]:
model.adata

#### Visualize Garfield Latent Space

In [ ]:
sc.settings.set_figure_params(dpi=100, facecolor='white')

sc.pl.umap(model.adata, color=['orig.ident', 'cell_type',
                               'latent_leiden_0.8'], ncols=1,
           s=5, show=False) # , legend_loc='on data'

In [ ]:
sc.pl.umap(model.adata, color=['cell_type'], ncols=1,
           s=5, show=False, legend_loc='on data') #

In [ ]:
# Save trained model
model_folder_path = f"{workdir}/model_ref"
os.makedirs(model_folder_path, exist_ok=True)

model.save(dir_path=model_folder_path,
           overwrite=True,
           save_adata=True,
           adata_file_name="adata_ref.h5ad")

In [ ]:
from Garfield.model import Garfield

workdir = f'/pri_exthome/zhouwg/project/Garfield_benchmark/results/sp_unimodal/spRNA_integrated'
gf.settings.set_workdir(workdir)
model_folder_path = f"{workdir}/model_ref"

model = Garfield.load(dir_path=model_folder_path,
                      adata_file_name="adata_ref.h5ad")

In [ ]:
adata_stereo

In [ ]:
# read spatial data
import scipy.sparse as sp

dataset='model_stereo'
dataset_name = 'stereo'
root_dir = '/pri_exthome/zhouwg/project/Garfield_benchmark/results/sp_unimodal/spRNA_integrated'
file_fold = os.path.join(root_dir, str(dataset))
tmp = sc.read_h5ad(file_fold + f'/adata_{dataset_name}.h5ad')
query_adata = adata_stereo.copy()
query_adata.obs['niche_type'] = tmp.obs['niche_type']
query_adata.var_names_make_unique()
query_adata

In [ ]:
# Ensure adata.X is counts.
# query_adata.layers['counts'] = query_adata.X.copy()
# adata.X = adata.layers['counts'].copy()
query_adata.X.max()

#### Integrating spatially resolved transcriptomics data using Garfield

In [ ]:
new_model = model.load_query_data(dir_path=model_folder_path,
                                  query_adata=query_adata,
                                  ref_adata_name="adata_ref.h5ad",
                                  use_cuda=True,
                                  unfreeze_all_weights=False,
                                  unfreeze_eps_weight=True,
                                  unfreeze_layer0=True,
                                  used_mmd=True,
                                  sample_col='projection',
                                  lambda_omics_recon_mmd_loss=1.0)
# Training and obtain latent representation
new_model.train()

# plot UMAP
sc.pp.neighbors(new_model.adata, use_rep='garfield_latent')
sc.tl.umap(new_model.adata)
sc.pl.umap(new_model.adata,
           color=['projection', 'cell_type','niche_type'],
           ncols=1, wspace=0.20, edges=False)

## split
adata_ref = new_model.adata[new_model.adata.obs['projection'] == 'reference', :]
adata_query = new_model.adata[new_model.adata.obs['projection'] == 'query', :]
spatial_tmp = query_adata.obsm['spatial'].copy()

In [ ]:
### Label transfer
## major celltype
adata_query = new_model.label_transfer(ref_adata=adata_ref,
                                       ref_adata_emb='garfield_latent',
                                       query_adata=adata_query,
                                       query_adata_emb='garfield_latent',
                                       n_neighbors=10,
                                       ref_adata_obs=adata_ref.obs,
                                       label_keys='cell_type')

In [ ]:
adata_query

In [ ]:
adata_query.obsm['spatial'] = spatial_tmp
sc.pl.embedding(adata_query, basis="spatial",
                color=["transferred_cell_type_unfiltered"],
                ncols=1, wspace=0.20, edges=False)

sc.pl.umap(adata_query, color=["transferred_cell_type_unfiltered"],
           ncols=1, wspace=0.20, edges=False)

In [ ]:
latent_cluster_key = 'niche_type'
cell_type_key = 'transferred_cell_type_unfiltered'

In [ ]:
adata_query

In [ ]:
import pandas as pd

def calculate_celltype_proportion(confusion_matrix, save_path=None, file_format='csv'):
    """
    计算每个niche中各个celltype的比例，并可选择保存结果。

    参数：
    - confusion_matrix (pd.DataFrame): 行是niches，列是celltypes的混淆矩阵。
    - save_path (str, optional): 保存结果的路径。如果为None，则不保存结果。
    - file_format (str, optional): 保存文件的格式，支持 'csv' 和 'xlsx'，默认为 'csv'。

    返回：
    - proportion_matrix (pd.DataFrame): 每个niche中各个celltype的比例矩阵。
    """
    # 计算每个niche的总数（行的总和）
    niche_totals = confusion_matrix.sum(axis=1)

    # 计算每个celltype在每个niche中的比例
    proportion_matrix = confusion_matrix.div(niche_totals, axis=0)

    # 如果指定了保存路径，保存结果
    if save_path:
        if file_format == 'csv':
            proportion_matrix.to_csv(save_path)
            print(f"结果已保存为 {save_path} (CSV 格式)")
        elif file_format == 'xlsx':
            proportion_matrix.to_excel(save_path)
            print(f"结果已保存为 {save_path} (Excel 格式)")
        else:
            print(f"不支持的文件格式 {file_format}，未保存结果")

    return proportion_matrix

# 调用函数并保存结果
df = adata_query.obs[[latent_cluster_key, cell_type_key]].groupby([latent_cluster_key, cell_type_key]).size().unstack(fill_value=0)
proportion_matrix = calculate_celltype_proportion(confusion_matrix=df,
                                                  save_path=f'{workdir}/niche_prop_stereo.csv',
                                                  file_format='csv')
# 输出 df
df.to_csv(f'{workdir}/niche_num_stereo.csv')

# 打印比例矩阵
proportion_matrix

In [ ]:
# scanpy结果画细胞分类百分比例图
import os
import scanpy as sc
import pandas as pd
import anndata as ad
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.spatial.distance import pdist, squareform

# 数据处理
df = adata_query.obs[[latent_cluster_key, cell_type_key]].groupby([latent_cluster_key, cell_type_key]).size().unstack(fill_value=0)
labels = df.index.tolist()  # 提取分类标签
results = df.to_dict(orient="list")  # 将数值结果转化为字典
category_names = list(results.keys())  # 提取类别（键-key）
data = np.array(list(results.values()))  # 提取数值（值-value）
category_colors = adata_query.uns["transferred_cell_type_unfiltered_colors"]  # 分类颜色

# 计算相似性并进行层次聚类
data_normalized = data / data.sum(axis=0)  # 正规化百分比
dist_matrix = squareform(pdist(data_normalized.T, metric="euclidean"))  # 计算样本之间的欧式距离
linkage_matrix = linkage(dist_matrix, method="ward")  # 使用层次聚类方法
dendro = dendrogram(linkage_matrix, labels=labels, no_plot=True)  # 获取聚类树的顺序

# 按聚类结果重新排序
ordered_labels = [labels[i] for i in dendro['leaves']]
ordered_data = data_normalized[:, dendro['leaves']]

# 创建主图和聚类树
fig = plt.figure(figsize=(8, 6))
gs = fig.add_gridspec(2, 1, height_ratios=[1, 4], hspace=0.05)

# 绘制聚类树
ax_dendro = fig.add_subplot(gs[0, 0])
dendrogram(
    linkage_matrix,
    labels=ordered_labels,
    orientation="top",
    leaf_rotation=90,
    ax=ax_dendro,
    color_threshold=0,
)
ax_dendro.axis("off")  # 隐藏聚类树的坐标轴

# 绘制柱状图
ax_bar = fig.add_subplot(gs[1, 0])
starts = np.zeros(len(ordered_labels))  # 基准为 0

for i, (colname, color) in enumerate(zip(category_names, category_colors)):
    heights = ordered_data[i, :]  # 使用聚类排序后的数据
    ax_bar.bar(
        ordered_labels,
        heights,
        bottom=starts,
        width=0.5,
        label=colname,
        color=color,
        edgecolor="gray",
    )
    starts += heights  # 基于基准累加

# 美化柱状图
ax_bar.tick_params(axis="x", length=0)
ax_bar.set_xlabel("Niches", fontsize=12)
ax_bar.set_ylabel("Proportion", fontsize=12)
ax_bar.set_ylim(0, 1.01)
ax_bar.set_yticks(np.arange(0, 1.2, 0.2))
ax_bar.set_yticklabels([f"{int(i * 100)}%" for i in np.arange(0, 1.2, 0.2)])
ax_bar.grid(axis="y", alpha=0.5, ls="--")

# 添加图例，将图例移至图外
plt.legend(title='Cluster', bbox_to_anchor=(1.05, 1), loc='upper left')

# 保存和展示
plt.tight_layout()
# plt.savefig(os.path.join(workdir, "05.barplot_niche.pdf"), dpi=600)
# pic(os.path.join(workdir, "05.barplot_niche.pdf"))

#### 保存 model 结果

In [ ]:
# Save trained model
model_folder_path = f"{workdir}/model_ref_stereo"
os.makedirs(model_folder_path, exist_ok=True)

new_model.save(dir_path=model_folder_path,
           overwrite=True,
           save_adata=True,
           adata_file_name="adata_concat_ref_stereo.h5ad")

In [ ]:
from Garfield.model import Garfield

workdir = f'/pri_exthome/zhouwg/project/Garfield_benchmark/results/sp_unimodal/spRNA_integrated'
gf.settings.set_workdir(workdir)
model_folder_path = f"{workdir}/model_ref_stereo"

model = Garfield.load(dir_path=model_folder_path,
                      adata_file_name="adata_concat_ref_stereo.h5ad")

In [ ]:
### 输出celltype 和 空间位置数据供R绘图
# Extracting 'latent_leiden_0.3' and 'spatial' data
cell_type_key = 'transferred_cell_type_unfiltered'
latent_leiden_data = adata_query.obs[cell_type_key]
niche_data = adata_query.obs['transferred_cell_type_uncert']
spatial_data = adata_query.obsm["spatial"]

# Creating a DataFrame for R visualization
output_df = pd.DataFrame({
    cell_type_key: latent_leiden_data,
    'transferred_cell_type_uncert': niche_data,
    "spatial_X": spatial_data[:, 0],
    "spatial_Y": spatial_data[:, 1],
})

# Save to CSV
output_file = f"latent_celltype_data_{dataset_name}.csv"
output_df.to_csv(os.path.join(workdir, output_file))

In [ ]:
model.adata